# PCA-Training

In diesem Notebook werden die zuvor generierten generischen und domänenspezifischen Vektor-Embeddings für die finale Evaluation vorbereitet, indem sie in **Trainings- und Evaluationsdatensätze (60/40-Split)** unterteilt und hinsichtlich ihrer Query-Zuordnungen bereinigt werden, um Data Leakage strikt auszuschließen. Darauf aufbauend werden jeweils **generische** und **domänenspezifische PCA-Pipelines**, die aus Standardisierung und PCA bestehen, isoliert auf den Trainingsdaten **trainiert**.

## Vorbereitung

Zunächst werden die benötigten Bibliotheken importiert.

In [2]:
import numpy as np
import joblib
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns 
import json

Anschließend werden die Dateipfade für den Datenzugriff definiert.

In [30]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"
# PCA Modelle
MODELS_DIR = BASE_DIR / "models"
# generisch
GENERIC_CORPUS_DIR  = DATA_DIR / "generic" / "corpus"
GENERIC_QUERIES_DIR = DATA_DIR / "generic" / "queries"  
GENERIC_IDS_DIR     = DATA_DIR / "generic" / "ids"  
GENERIC_SPLIT_DIR   = DATA_DIR / "generic" / "splits"
# medizinisch
MED_CORPUS_DIR = DATA_DIR / "medicine" / "corpus"
MED_QUERIES_DIR = DATA_DIR / "medicine" / "queries"
MED_IDS_DIR = DATA_DIR / "medicine" / "ids"
MED_SPLIT_DIR   = DATA_DIR / "medicine" / "splits"
# recht
LAW_CORPUS_DIR = DATA_DIR / "law" / "corpus"
LAW_QUERIES_DIR = DATA_DIR / "law" / "queries"
LAW_IDS_DIR = DATA_DIR / "law" / "ids"
LAW_SPLIT_DIR   = DATA_DIR / "law" / "splits"
# finanzen
FIN_CORPUS_DIR = DATA_DIR / "finance" / "corpus"
FIN_QUERIES_DIR = DATA_DIR / "finance" / "queries"
FIN_IDS_DIR = DATA_DIR / "finance" / "ids"
FIN_SPLIT_DIR   = DATA_DIR / "finance" / "splits"

## PCA-Training auf dem generischen Datensatz (NQ)

Zunächst wird der Startwert des Zufallsgenerators fixiert, um die Reproduzierbarkeit zu gewährleisten.

In [5]:
SEED = 42

Anschließend werden die zuvor berechneten Embedding-Vektoren, IDs und Relevanzurteile (QRels) geladen. 

In [6]:
print("Lade generische Daten...")
generic_corpus_vecs = np.load(GENERIC_CORPUS_DIR / "generic_corpus_vecs.npy")
generic_query_vecs  = np.load(GENERIC_QUERIES_DIR / "generic_query_vecs.npy")
generic_corpus_ids  = np.load(GENERIC_IDS_DIR / "generic_corpus_ids.npy", allow_pickle=True)
generic_query_ids   = np.load(GENERIC_IDS_DIR / "generic_query_ids.npy", allow_pickle=True)

with open(GENERIC_IDS_DIR / "generic_qrels.json", "r") as f:
    generic_qrels = json.load(f)

print(f"  Corpus:  {generic_corpus_vecs.shape}")
print(f"  Queries: {generic_query_vecs.shape}")
print(f"  QRels:   {len(generic_qrels)} Query-Zuordnungen")

Lade generische Daten...
  Corpus:  (100000, 384)
  Queries: (2924, 384)
  QRels:   2924 Query-Zuordnungen


Die 100.000 Corpus-Dokumente werden zufällig im Verhältnis **60/40** aufgeteilt.

In [7]:
n_corpus = len(generic_corpus_ids)
indices = np.arange(n_corpus)

train_idx, eval_idx = train_test_split(
    indices,
    train_size=0.6,
    random_state=SEED,
    shuffle=True
)

train_vecs       = generic_corpus_vecs[train_idx]
train_ids        = generic_corpus_ids[train_idx]
eval_corpus_vecs = generic_corpus_vecs[eval_idx]
eval_corpus_ids  = generic_corpus_ids[eval_idx]

print(f"Training:   {train_vecs.shape[0]:,} Dokumente (60%)")
print(f"Evaluation: {eval_corpus_vecs.shape[0]:,} Dokumente (40%)")

Training:   60,000 Dokumente (60%)
Evaluation: 40,000 Dokumente (40%)


Nicht jede Query hat ihr relevantes Dokument im Evaluations-Split. Nur Queries werden behalten, für die mindestens ein relevantes Dokument in den 40% Eval-Dokumenten liegt.

In [8]:
eval_corpus_id_set = set(eval_corpus_ids.tolist())

# QRels filtern, nur relevante Docs werden behalten, die im Eval-Split liegen
eval_qrels = {}
for q_id, rel_doc_ids in generic_qrels.items():
    rel_in_eval = [d for d in rel_doc_ids if d in eval_corpus_id_set]
    if rel_in_eval:
        eval_qrels[q_id] = rel_in_eval

# Queries filtern
eval_query_ids_set = set(eval_qrels.keys())
query_mask = np.array([str(qid) in eval_query_ids_set for qid in generic_query_ids])

eval_query_vecs = generic_query_vecs[query_mask]
eval_query_ids  = generic_query_ids[query_mask]

print(f"Eval-Queries: {len(eval_query_ids):,} (von {len(generic_query_ids):,})")
print(f"Eval-QRels:   {sum(len(v) for v in eval_qrels.values()):,} Zuordnungen")

Eval-Queries: 1,331 (von 2,924)
Eval-QRels:   1,431 Zuordnungen


Alle Split-Indizes und Eval-Daten werden gespeichert, damit der Split bei der späteren Evaluation exakt reproduziert werden kann. Folgende Dateien werden persistiert:

* **Split-Indizes (`train_indices.npy`, `eval_indices.npy`):** Positionsindizes, die festlegen, welche Dokumente für das PCA-Training bzw. die Evaluation verwendet werden.
* **Eval-Vektoren (`eval_corpus_vecs.npy`, `eval_query_vecs.npy`):** Die Embedding-Vektoren der Evaluations-Dokumente und -Queries als Input für die PCA-Transformation.
* **Eval-IDs (`eval_corpus_ids.npy`, `eval_query_ids.npy`):** Zuordnung von Vektorpositionen zu Dokumenten- bzw. Query-Namen für die spätere Ergebnisauswertung.
* **Ground Truth (`eval_qrels.json`):** JSON-Dictionary, das für jede Query die relevanten Dokument-IDs enthält (ausschließlich für Eval-Paare).
* **Metadaten (`split_info.json`):** Seed, Split-Verhältnis und Datengrößen zur Reproduzierbarkeit.


In [13]:
# Split-Indizes
np.save(GENERIC_SPLIT_DIR / "train_indices.npy", train_idx)
np.save(GENERIC_SPLIT_DIR / "eval_indices.npy", eval_idx)

# Eval-Vektoren und IDs
np.save(GENERIC_SPLIT_DIR / "eval_corpus_vecs.npy", eval_corpus_vecs)
np.save(GENERIC_SPLIT_DIR / "eval_corpus_ids.npy", eval_corpus_ids)
np.save(GENERIC_SPLIT_DIR / "eval_query_vecs.npy", eval_query_vecs)
np.save(GENERIC_SPLIT_DIR / "eval_query_ids.npy", eval_query_ids)

# Ground Truth
with open(GENERIC_SPLIT_DIR / "eval_qrels.json", "w") as f:
    json.dump(eval_qrels, f)

# Zusammenfassung
split_info = {
    "seed": SEED,
    "train_size": 0.6,
    "n_corpus_total": int(n_corpus),
    "n_corpus_train": int(len(train_idx)),
    "n_corpus_eval": int(len(eval_idx)),
    "n_queries_total": int(len(generic_query_ids)),
    "n_queries_eval": int(len(eval_query_ids)),
    "n_qrels_eval": int(sum(len(v) for v in eval_qrels.values()))
}
with open(GENERIC_SPLIT_DIR / "split_info.json", "w") as f:
    json.dump(split_info, f, indent=2)

print("Alle Split-Daten gespeichert unter:", GENERIC_SPLIT_DIR)

Alle Split-Daten gespeichert unter: /mnt/user/experiment/data/generic/splits


Die Pipeline besteht aus `StandardScaler` (Zentrierung + Normierung) und `PCA` (384 Komponenten). Sie wird **ausschließlich** auf den 60% Train-Vektoren trainiert.

In [14]:
pipeline_generic = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=384))
])

print(f"Trainiere PCA auf {train_vecs.shape[0]:,} Vektoren ({train_vecs.shape[1]} Dim)...")
pipeline_generic.fit(train_vecs)

joblib.dump(pipeline_generic, MODELS_DIR / "pca_pipeline_generic.joblib")
print("Pipeline gespeichert:", MODELS_DIR / "pca_pipeline_generic.joblib")

Trainiere PCA auf 60,000 Vektoren (384 Dim)...
Pipeline gespeichert: /mnt/user/experiment/models/pca_pipeline_generic.joblib


## PCA-Training auf dem medizinischen Datensatz (PubMedQA)

Nun werden die Embeddings der **medizinischen** Domäne geladen.

In [15]:
SEED = 42

print("Lade medizinische Daten...")
med_corpus_vecs = np.load(MED_CORPUS_DIR / "medicine_corpus_vecs.npy")
med_query_vecs  = np.load(MED_QUERIES_DIR / "medicine_query_vecs.npy")
med_corpus_ids  = np.load(MED_IDS_DIR / "medicine_corpus_ids.npy", allow_pickle=True)
med_query_ids   = np.load(MED_IDS_DIR / "medicine_query_ids.npy", allow_pickle=True)

print(f"  Corpus:  {med_corpus_vecs.shape}")
print(f"  Queries: {med_query_vecs.shape}")

Lade medizinische Daten...
  Corpus:  (1000, 384)
  Queries: (1000, 384)


Bei PubMedQA ist die Query-ID identisch mit der Doc-ID (`pub_id`). Jede Frage gehört zu genau einem Dokument (1:1-Mapping).

In [16]:
# Jede Query-ID == Doc-ID (1:1 Mapping)
med_qrels = {str(qid): [str(qid)] for qid in med_query_ids}

print(f"QRels erzeugt: {len(med_qrels)} Query-Dokument-Paare")
print(f"Beispiel: Query '{list(med_qrels.keys())[0]}' → Doc {med_qrels[list(med_qrels.keys())[0]]}")

QRels erzeugt: 1000 Query-Dokument-Paare
Beispiel: Query '21645374' → Doc ['21645374']


Die 1.000 Corpus-Dokumente werden zufällig im Verhältnis **60/40** aufgeteilt.

In [17]:
n_corpus = len(med_corpus_ids)
indices = np.arange(n_corpus)

train_idx, eval_idx = train_test_split(
    indices,
    train_size=0.6,
    random_state=SEED,
    shuffle=True
)

train_vecs       = med_corpus_vecs[train_idx]
train_ids        = med_corpus_ids[train_idx]
eval_corpus_vecs = med_corpus_vecs[eval_idx]
eval_corpus_ids  = med_corpus_ids[eval_idx]

print(f"Training:   {train_vecs.shape[0]:,} Dokumente (60%)")
print(f"Evaluation: {eval_corpus_vecs.shape[0]:,} Dokumente (40%)")


Training:   600 Dokumente (60%)
Evaluation: 400 Dokumente (40%)


Nur Queries werden behalten, deren relevantes Dokument im 40%-Eval-Split liegt.

In [18]:
eval_corpus_id_set = set(eval_corpus_ids.tolist())

# QRels filtern
eval_qrels = {}
for q_id, rel_doc_ids in med_qrels.items():
    rel_in_eval = [d for d in rel_doc_ids if d in eval_corpus_id_set]
    if rel_in_eval:
        eval_qrels[q_id] = rel_in_eval

# Queries filtern
eval_query_ids_set = set(eval_qrels.keys())
query_mask = np.array([str(qid) in eval_query_ids_set for qid in med_query_ids])

eval_query_vecs = med_query_vecs[query_mask]
eval_query_ids  = med_query_ids[query_mask]

print(f"Eval-Queries: {len(eval_query_ids):,} (von {len(med_query_ids):,})")
print(f"Eval-QRels:   {sum(len(v) for v in eval_qrels.values()):,} Zuordnungen")

Eval-Queries: 400 (von 1,000)
Eval-QRels:   400 Zuordnungen


Analog zum generischen Split werden die gleichen Artefakte für die Medizin-Domäne persistiert.

In [19]:
MED_SPLIT_DIR = DATA_DIR / "medicine" / "splits"
MED_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

# Split-Indizes
np.save(MED_SPLIT_DIR / "train_indices.npy", train_idx)
np.save(MED_SPLIT_DIR / "eval_indices.npy", eval_idx)

# Eval-Vektoren und IDs
np.save(MED_SPLIT_DIR / "eval_corpus_vecs.npy", eval_corpus_vecs)
np.save(MED_SPLIT_DIR / "eval_corpus_ids.npy", eval_corpus_ids)
np.save(MED_SPLIT_DIR / "eval_query_vecs.npy", eval_query_vecs)
np.save(MED_SPLIT_DIR / "eval_query_ids.npy", eval_query_ids)

# Ground Truth
with open(MED_SPLIT_DIR / "eval_qrels.json", "w") as f:
    json.dump(eval_qrels, f)

# Zusammenfassung
split_info = {
    "seed": SEED,
    "train_size": 0.6,
    "n_corpus_total": int(n_corpus),
    "n_corpus_train": int(len(train_idx)),
    "n_corpus_eval": int(len(eval_idx)),
    "n_queries_total": int(len(med_query_ids)),
    "n_queries_eval": int(len(eval_query_ids)),
    "n_qrels_eval": int(sum(len(v) for v in eval_qrels.values()))
}
with open(MED_SPLIT_DIR / "split_info.json", "w") as f:
    json.dump(split_info, f, indent=2)

print("Alle Split-Daten gespeichert unter:", MED_SPLIT_DIR)


Alle Split-Daten gespeichert unter: /mnt/user/experiment/data/medicine/splits


Die Pipeline besteht aus `StandardScaler` (Zentrierung + Normierung) und `PCA` (384 Komponenten). Sie wird **ausschließlich** auf den 60% Train-Vektoren trainiert.

In [20]:
pipeline_med = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=384))
])

print(f"Trainiere PCA auf {train_vecs.shape[0]:,} Vektoren ({train_vecs.shape[1]} Dim)...")
pipeline_med.fit(train_vecs)

joblib.dump(pipeline_med, MODELS_DIR / "pca_pipeline_medicine.joblib")
print("Pipeline gespeichert:", MODELS_DIR / "pca_pipeline_medicine.joblib")

Trainiere PCA auf 600 Vektoren (384 Dim)...
Pipeline gespeichert: /mnt/user/experiment/models/pca_pipeline_medicine.joblib


## PCA-Training auf dem juristischen Datensatz (LegalBenchRAG)

Weiterhin werden die Embeddings der **rechtspezifischen** Domäne geladen.

In [22]:
SEED = 42

print("Lade juristische Daten...")
law_corpus_vecs = np.load(LAW_CORPUS_DIR / "law_corpus_vecs.npy")
law_query_vecs  = np.load(LAW_QUERIES_DIR / "law_query_vecs.npy")
law_corpus_ids  = np.load(LAW_IDS_DIR / "law_corpus_ids.npy", allow_pickle=True)
law_query_ids   = np.load(LAW_IDS_DIR / "law_query_ids.npy", allow_pickle=True)

print(f"  Corpus:  {law_corpus_vecs.shape}")
print(f"  Queries: {law_query_vecs.shape}")
print(f"  Unique Corpus-Docs: {len(set(law_corpus_ids))}")
print(f"  Unique Query-Targets: {len(set(law_query_ids))}")

Lade juristische Daten...
  Corpus:  (698, 384)
  Queries: (6889, 384)
  Unique Corpus-Docs: 698
  Unique Query-Targets: 714


Bei LegalBenchRAG ergibt sich die Ground-Truth-Zuordnung direkt aus `law_query_ids[i]`, dem Dateipfad des relevanten Dokuments für Query `i`.
Als Query-ID dient der laufende Index, als Doc-ID der Dateipfad.

In [23]:
# Query-Index -> relevante Doc-ID(s)
law_qrels = {}
for i, doc_id in enumerate(law_query_ids):
    q_key = str(i)
    if q_key not in law_qrels:
        law_qrels[q_key] = []
    law_qrels[q_key].append(str(doc_id))

# Duplikate entfernen (falls eine Query mehrere Snippets aus demselben Doc hat)
law_qrels = {k: list(set(v)) for k, v in law_qrels.items()}

print(f"QRels erzeugt: {len(law_qrels)} Queries")
print(f"Beispiel: Query '0' → Doc {law_qrels['0']}")

QRels erzeugt: 6889 Queries
Beispiel: Query '0' → Doc ['contractnli/CopAcc_NDA-and-ToP-Mentors_2.0_2017.txt']


Die 698 Corpus-Dokumente werden zufällig im Verhältnis **60/40** aufgeteilt.

In [24]:
n_corpus = len(law_corpus_ids)
indices = np.arange(n_corpus)

train_idx, eval_idx = train_test_split(
    indices,
    train_size=0.6,
    random_state=SEED,
    shuffle=True
)

train_vecs       = law_corpus_vecs[train_idx]
train_ids        = law_corpus_ids[train_idx]
eval_corpus_vecs = law_corpus_vecs[eval_idx]
eval_corpus_ids  = law_corpus_ids[eval_idx]

print(f"Training:   {train_vecs.shape[0]:,} Dokumente (60%)")
print(f"Evaluation: {eval_corpus_vecs.shape[0]:,} Dokumente (40%)")

Training:   418 Dokumente (60%)
Evaluation: 280 Dokumente (40%)


Nur Queries werden behalten, deren relevantes Dokument im 40%-Eval-Split liegt.

In [25]:
eval_corpus_id_set = set(eval_corpus_ids.tolist())

# QRels filtern
eval_qrels = {}
for q_id, rel_doc_ids in law_qrels.items():
    rel_in_eval = [d for d in rel_doc_ids if d in eval_corpus_id_set]
    if rel_in_eval:
        eval_qrels[q_id] = rel_in_eval

# Queries filtern (Query-ID = laufender Index)
eval_query_indices = sorted([int(q) for q in eval_qrels.keys()])
eval_query_vecs = law_query_vecs[eval_query_indices]
eval_query_ids  = np.array([str(i) for i in eval_query_indices])

print(f"Eval-Queries: {len(eval_query_ids):,} (von {len(law_query_ids):,})")
print(f"Eval-QRels:   {sum(len(v) for v in eval_qrels.values()):,} Zuordnungen")

Eval-Queries: 2,710 (von 6,889)
Eval-QRels:   2,710 Zuordnungen


Analog werden die gleichen Artefakte für die Recht-Domäne persistiert.

In [27]:
LAW_SPLIT_DIR = DATA_DIR / "law" / "splits"
LAW_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

# Split-Indizes
np.save(LAW_SPLIT_DIR / "train_indices.npy", train_idx)
np.save(LAW_SPLIT_DIR / "eval_indices.npy", eval_idx)

# Eval-Vektoren und IDs
np.save(LAW_SPLIT_DIR / "eval_corpus_vecs.npy", eval_corpus_vecs)
np.save(LAW_SPLIT_DIR / "eval_corpus_ids.npy", eval_corpus_ids)
np.save(LAW_SPLIT_DIR / "eval_query_vecs.npy", eval_query_vecs)
np.save(LAW_SPLIT_DIR / "eval_query_ids.npy", eval_query_ids)

# Ground Truth
with open(LAW_SPLIT_DIR / "eval_qrels.json", "w") as f:
    json.dump(eval_qrels, f)

# Zusammenfassung
split_info = {
    "seed": SEED,
    "train_size": 0.6,
    "n_corpus_total": int(n_corpus),
    "n_corpus_train": int(len(train_idx)),
    "n_corpus_eval": int(len(eval_idx)),
    "n_queries_total": int(len(law_query_ids)),
    "n_queries_eval": int(len(eval_query_ids)),
    "n_qrels_eval": int(sum(len(v) for v in eval_qrels.values()))
}
with open(LAW_SPLIT_DIR / "split_info.json", "w") as f:
    json.dump(split_info, f, indent=2)

print("Alle Split-Daten gespeichert unter:", LAW_SPLIT_DIR)


Alle Split-Daten gespeichert unter: /mnt/user/experiment/data/law/splits


Die Pipeline besteht aus `StandardScaler` (Zentrierung + Normierung) und `PCA` (384 Komponenten). Sie wird **ausschließlich** auf den 60% Train-Vektoren trainiert.

In [29]:
pipeline_law = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=384))
])

print(f"Trainiere PCA auf {train_vecs.shape[0]:,} Vektoren ({train_vecs.shape[1]} Dim)...")
pipeline_law.fit(train_vecs)

joblib.dump(pipeline_law, MODELS_DIR / "pca_pipeline_law.joblib")
print("Pipeline gespeichert:", MODELS_DIR / "pca_pipeline_law.joblib")

Trainiere PCA auf 418 Vektoren (384 Dim)...
Pipeline gespeichert: /mnt/user/experiment/models/pca_pipeline_law.joblib


## PCA-Training auf dem Finanz-Datensatz (FinanceBench)

Abschließend werden die Embeddings für die **finanzspezifische** Domäne geladen.

In [31]:
SEED = 42

print("Lade Finanz-Daten...")
fin_corpus_vecs = np.load(FIN_CORPUS_DIR / "finance_corpus_vecs.npy")
fin_query_vecs  = np.load(FIN_QUERIES_DIR / "finance_query_vecs.npy")
fin_corpus_ids  = np.load(FIN_IDS_DIR / "finance_corpus_ids.npy", allow_pickle=True)
fin_query_ids   = np.load(FIN_IDS_DIR / "finance_query_ids.npy", allow_pickle=True)

print(f"  Corpus:  {fin_corpus_vecs.shape}")
print(f"  Queries: {fin_query_vecs.shape}")
print(f"  Kleiner Datensatz — nur {fin_corpus_vecs.shape[0]} Dokumente")

Lade Finanz-Daten...
  Corpus:  (150, 384)
  Queries: (150, 384)
  Kleiner Datensatz — nur 150 Dokumente


Bei FinanceBench, genau wie bei PubMedQA, ist die Query-ID identisch mit der Doc-ID (`financebench_id`). Jede Frage gehört zu genau einem Dokument (1:1-Mapping).

In [32]:
# Jede Query-ID == Doc-ID -> 1:1 Mapping
fin_qrels = {str(qid): [str(qid)] for qid in fin_query_ids}

print(f"QRels erzeugt: {len(fin_qrels)} Query-Dokument-Paare")
print(f"Beispiel: Query '{list(fin_qrels.keys())[0]}' → Doc {fin_qrels[list(fin_qrels.keys())[0]]}")

QRels erzeugt: 150 Query-Dokument-Paare
Beispiel: Query 'financebench_id_03029' → Doc ['financebench_id_03029']


Die 150 Corpus-Dokumente werden zufällig im Verhältnis **60/40** aufgeteilt.

In [33]:
n_corpus = len(fin_corpus_ids)
indices = np.arange(n_corpus)

train_idx, eval_idx = train_test_split(
    indices,
    train_size=0.6,
    random_state=SEED,
    shuffle=True
)

train_vecs       = fin_corpus_vecs[train_idx]
train_ids        = fin_corpus_ids[train_idx]
eval_corpus_vecs = fin_corpus_vecs[eval_idx]
eval_corpus_ids  = fin_corpus_ids[eval_idx]

print(f"Training:   {train_vecs.shape[0]:,} Dokumente (60%)")
print(f"Evaluation: {eval_corpus_vecs.shape[0]:,} Dokumente (40%)")

Training:   90 Dokumente (60%)
Evaluation: 60 Dokumente (40%)


Nur Queries werden behalten, deren relevantes Dokument im 40%-Eval-Split liegt.

In [34]:
eval_corpus_id_set = set(eval_corpus_ids.tolist())

# QRels filtern
eval_qrels = {}
for q_id, rel_doc_ids in fin_qrels.items():
    rel_in_eval = [d for d in rel_doc_ids if d in eval_corpus_id_set]
    if rel_in_eval:
        eval_qrels[q_id] = rel_in_eval

# Queries filtern
eval_query_ids_set = set(eval_qrels.keys())
query_mask = np.array([str(qid) in eval_query_ids_set for qid in fin_query_ids])

eval_query_vecs = fin_query_vecs[query_mask]
eval_query_ids  = fin_query_ids[query_mask]

print(f"Eval-Queries: {len(eval_query_ids):,} (von {len(fin_query_ids):,})")
print(f"Eval-QRels:   {sum(len(v) for v in eval_qrels.values()):,} Zuordnungen")

Eval-Queries: 60 (von 150)
Eval-QRels:   60 Zuordnungen


Analog werden die gleichen Artefakte für die Finanz-Domäne persistiert.

In [35]:
FIN_SPLIT_DIR = DATA_DIR / "finance" / "splits"
FIN_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

# Split-Indizes
np.save(FIN_SPLIT_DIR / "train_indices.npy", train_idx)
np.save(FIN_SPLIT_DIR / "eval_indices.npy", eval_idx)

# Eval-Vektoren und IDs
np.save(FIN_SPLIT_DIR / "eval_corpus_vecs.npy", eval_corpus_vecs)
np.save(FIN_SPLIT_DIR / "eval_corpus_ids.npy", eval_corpus_ids)
np.save(FIN_SPLIT_DIR / "eval_query_vecs.npy", eval_query_vecs)
np.save(FIN_SPLIT_DIR / "eval_query_ids.npy", eval_query_ids)

# Ground Truth
with open(FIN_SPLIT_DIR / "eval_qrels.json", "w") as f:
    json.dump(eval_qrels, f)

# Zusammenfassung
split_info = {
    "seed": SEED,
    "train_size": 0.6,
    "n_corpus_total": int(n_corpus),
    "n_corpus_train": int(len(train_idx)),
    "n_corpus_eval": int(len(eval_idx)),
    "n_queries_total": int(len(fin_query_ids)),
    "n_queries_eval": int(len(eval_query_ids)),
    "n_qrels_eval": int(sum(len(v) for v in eval_qrels.values()))
}
with open(FIN_SPLIT_DIR / "split_info.json", "w") as f:
    json.dump(split_info, f, indent=2)

print("Alle Split-Daten gespeichert unter:", FIN_SPLIT_DIR)

Alle Split-Daten gespeichert unter: /mnt/user/experiment/data/finance/splits


Die Pipeline besteht aus `StandardScaler` (Zentrierung + Normierung) und `PCA` (384 Komponenten). Sie wird **ausschließlich** auf den 60% Train-Vektoren trainiert.<br>
**Einschränkung:** PCA erlaubt maximal `min(n_samples, n_features)` Komponenten. Bei nur 90 Train-Dokumenten können maximal **90 statt 384** Komponenten extrahiert werden.

In [37]:
max_components = min(train_vecs.shape[0], train_vecs.shape[1])

pipeline_fin = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=max_components))
])

print(f"Trainiere PCA auf {train_vecs.shape[0]:,} Vektoren ({train_vecs.shape[1]} Dim)...")
print(f"Maximal mögliche Komponenten: {max_components} (statt 384)")
pipeline_fin.fit(train_vecs)

joblib.dump(pipeline_fin, MODELS_DIR / "pca_pipeline_finance.joblib")
print("Pipeline gespeichert:", MODELS_DIR / "pca_pipeline_finance.joblib")


Trainiere PCA auf 90 Vektoren (384 Dim)...
Maximal mögliche Komponenten: 90 (statt 384)
Pipeline gespeichert: /mnt/user/experiment/models/pca_pipeline_finance.joblib
